In [4]:
# ============================================================
# PACKAGE 10.1 VERSION 2.0
# GEOSPATIAL POLLUTION SOURCE ATTRIBUTION ENGINE
#
# Purpose:
#   Attribute AQI prediction contribution into pollution sources
#   using SHAP explainability from production XGBoost model.
#
# Inputs:
#   - Production_XGBoost_Model.pkl
#   - X_test_selected.csv
#
# Outputs:
#   - Source contribution reports
#   - Confidence scores
#   - Frontend JSON
#   - Visualizations
#
# ============================================================


# ============================================================
# IMPORT LIBRARIES
# ============================================================


import os
import json
import joblib
import shap
import pandas as pd
import numpy as np

from datetime import datetime

import matplotlib.pyplot as plt


print("="*75)
print("PACKAGE 10.1 VERSION 2.0")
print("GEOSPATIAL POLLUTION SOURCE ATTRIBUTION ENGINE")
print("="*75)



# ============================================================
# CONFIGURATION
# ============================================================


MODEL_PATH = (
    r"C:\Users\91863\Desktop\AIR QUALITY INTELLIGENCE\Last_Stage\Final_XGBoost_Production_Training\Final_XGBoost_AQI_Model.pkl"
)


X_TEST_PATH = (
    r"C:\Users\91863\Desktop\AIR QUALITY INTELLIGENCE"
    r"\Forecasting Model Pipeline"
    r"\Package4_Feature_Selection"
    r"\X_test_selected.csv"
)



OUTPUT_FOLDER = r"C:\Users\91863\Desktop\AIR QUALITY INTELLIGENCE\Geospatial_Pollution_Source_Attribution_Engine\Geospatial Source Attribution Engine Output"


os.makedirs(
    OUTPUT_FOLDER,
    exist_ok=True
)



print("\nConfiguration Loaded")



# ============================================================
# LOAD TEST DATA
# ============================================================


print("\nLoading Test Dataset...")


X_test = pd.read_csv(
    X_TEST_PATH
)



print("\nDataset Shape")

print(
    X_test.shape
)



print("\nFeature Count:")

print(
    len(X_test.columns)
)



# ============================================================
# LOAD PRODUCTION MODEL
# ============================================================


print("\nLoading Production XGBoost Model...")


model = joblib.load(
    MODEL_PATH
)


print(
    "Model Loaded Successfully"
)



# ============================================================
# SHAP EXPLAINABILITY
# ============================================================


print("\nCalculating SHAP Values...")


explainer = shap.TreeExplainer(
    model
)


shap_values = explainer.shap_values(
    X_test
)


print(
    "SHAP Calculation Completed"
)



# ============================================================
# SHAP DATAFRAME
# ============================================================


shap_abs = pd.DataFrame(

    np.abs(shap_values),

    columns=X_test.columns

)



print("\nSHAP Matrix Shape")

print(
    shap_abs.shape
)



# ============================================================
# POLLUTION SOURCE FEATURE MAPPING
# ============================================================


SOURCE_MAPPING = {


    "Traffic Emissions":

    [

        "NO",

        "NO2",

        "NOx",

        "CO",

        "Benzene",

        "Toluene",

        "Xylene"

    ],



    "Particulate Matter":

    [

        "PM2.5",

        "PM10",

        "PM25_Diff",

        "PM10_Diff",

        "PM25_MA3",

        "PM10_MA3",

        "PM25_MA7",

        "PM10_MA7",

        "PM25_PM10_Ratio"

    ],



    "Meteorological Conditions":

    [

        "Rainfall",

        "Humidity",

        "Humidity_Change",

        "Temperature",

        "Temperature_Change",

        "Wind_Speed",

        "Wind_Change",

        "Wind_Lag1",

        "Wind_Humidity_Index"

    ],



    "Satellite Aerosols":

    [

        "MODIS_AOD",

        "MODIS_AOD_MA7",

        "Sentinel_AOD",

        "Sentinel_AOD_MA7"

    ],



    "Industrial Pollution":

    [

        "SO2",

        "NH3",

        "Industrial_Count",

        "Industrial_Area",

        "Industry_Distance"

    ],



    "Construction Activity":

    [

        "Construction_Count",

        "Construction_Area",

        "Construction_Density"

    ],



    "Road Network":

    [

        "Road_Length",

        "Road_Density",

        "Intersection_Count",

        "Road_Proximity"

    ],



    "AQI Historical Pattern":

    [

        "AQI",

        "AQI_Lag1",

        "AQI_Lag3",

        "AQI_Lag7",

        "AQI_MA3",

        "AQI_MA7"

    ]

}




# ============================================================
# AUTOMATIC FEATURE DETECTION
# ============================================================


print("\nDetecting Available Features...")


available_features = set(
    X_test.columns
)



source_available_features = {}



for source, features in SOURCE_MAPPING.items():


    detected = [

        feature

        for feature in features

        if feature in available_features

    ]


    if len(detected) > 0:


        source_available_features[source] = detected



print("\nDetected Pollution Sources")



for source, features in source_available_features.items():

    print(
        f"{source}: {features}"
    )



# ============================================================
# SOURCE CONTRIBUTION CALCULATION
# ============================================================


print("\nCalculating Source Contributions...")



source_scores = {}

source_feature_scores = {}



for source, features in source_available_features.items():


    feature_scores = {}


    total_score = 0



    for feature in features:


        score = shap_abs[feature].sum()


        feature_scores[feature] = score


        total_score += score



    source_scores[source] = total_score


    source_feature_scores[source] = feature_scores




print(
    "Source Contribution Calculation Completed"
)

# ============================================================
# NORMALIZE SOURCE CONTRIBUTIONS
# ============================================================


print("\nNormalizing Source Contributions...")


total_contribution = sum(
    source_scores.values()
)



source_percentage = {}



for source, score in source_scores.items():

    percentage = (
        score / total_contribution
    ) * 100


    source_percentage[source] = percentage



# ============================================================
# SOURCE CONTRIBUTION DATAFRAME
# ============================================================


contribution_df = pd.DataFrame(

    {

        "Pollution_Source":
        list(source_percentage.keys()),


        "Contribution_%":
        list(source_percentage.values())

    }

)



contribution_df = contribution_df.sort_values(

    by="Contribution_%",

    ascending=False

).reset_index(drop=True)



contribution_df.insert(

    0,

    "Rank",

    range(
        1,
        len(contribution_df)+1
    )

)



print("\n============================================================")
print("POLLUTION SOURCE CONTRIBUTION")
print("============================================================")


print(
    contribution_df
)



# ============================================================
# CONFIDENCE SCORE ENGINE
# ============================================================


def calculate_confidence(value):


    if value >= 40:

        return "Very High"


    elif value >= 25:

        return "High"


    elif value >= 15:

        return "Medium"


    elif value >= 5:

        return "Low"


    else:

        return "Very Low"




confidence_df = contribution_df.copy()



confidence_df["Confidence"] = (

    confidence_df["Contribution_%"]

    .apply(
        calculate_confidence
    )

)



# ============================================================
# SOURCE FEATURE DETAIL REPORT
# ============================================================


feature_rows = []



for source, features in source_feature_scores.items():


    for feature, score in features.items():


        feature_rows.append(

            {

            "Pollution_Source":
            source,


            "Feature":
            feature,


            "SHAP_Contribution":
            score

            }

        )




feature_report_df = pd.DataFrame(

    feature_rows

)



feature_report_df = feature_report_df.sort_values(

    by="SHAP_Contribution",

    ascending=False

)



# ============================================================
# FINAL ATTRIBUTION REPORT
# ============================================================


final_report = contribution_df.merge(

    confidence_df[

        [

        "Pollution_Source",

        "Confidence"

        ]

    ],

    on="Pollution_Source"

)



# ============================================================
# DOMINANT SOURCE
# ============================================================


dominant_source = (

    final_report.iloc[0]

)



print("\n============================================================")
print("DOMINANT POLLUTION SOURCE")
print("============================================================")


print(

    "Source :",

    dominant_source["Pollution_Source"]

)


print(

    "Contribution :",

    round(

        dominant_source["Contribution_%"],

        2

    ),

    "%"

)


print(

    "Confidence :",

    dominant_source["Confidence"]

)



# ============================================================
# SAVE CSV FILES
# ============================================================


print("\nSaving Reports...")



contribution_df.to_csv(

    os.path.join(

        OUTPUT_FOLDER,

        "Pollution_Source_Contribution.csv"

    ),

    index=False

)



confidence_df[

    [

    "Pollution_Source",

    "Confidence"

    ]

].to_csv(

    os.path.join(

        OUTPUT_FOLDER,

        "Source_Confidence_Scores.csv"

    ),

    index=False

)




final_report.to_csv(

    os.path.join(

        OUTPUT_FOLDER,

        "Source_Attribution_Report.csv"

    ),

    index=False

)



feature_report_df.to_csv(

    os.path.join(

        OUTPUT_FOLDER,

        "Feature_Level_Source_Contribution.csv"

    ),

    index=False

)



# ============================================================
# FRONTEND JSON GENERATION
# ============================================================


frontend_output = {


    "model":

    "Production XGBoost AQI Forecasting Model",


    "generated_on":

    str(datetime.now()),



    "dominant_source":

    dominant_source["Pollution_Source"],



    "dominant_source_confidence":

    dominant_source["Confidence"],



    "source_contribution":

    final_report.to_dict(

        orient="records"

    )

}




with open(

    os.path.join(

        OUTPUT_FOLDER,

        "Frontend_Source_Attribution.json"

    ),

    "w"

) as file:


    json.dump(

        frontend_output,

        file,

        indent=4

    )



# ============================================================
# BAR CHART
# ============================================================


plt.figure(

    figsize=(10,6)

)



plt.bar(

    contribution_df["Pollution_Source"],

    contribution_df["Contribution_%"]

)



plt.xticks(

    rotation=45,

    ha="right"

)



plt.ylabel(

    "Contribution (%)"

)


plt.title(

    "Pollution Source Contribution Analysis"

)


plt.tight_layout()



plt.savefig(

    os.path.join(

        OUTPUT_FOLDER,

        "Source_Contribution_Bar.png"

    ),

    dpi=300

)



plt.close()



# ============================================================
# PIE CHART
# ============================================================


plt.figure(

    figsize=(8,8)

)



plt.pie(

    contribution_df["Contribution_%"],

    labels=

    contribution_df["Pollution_Source"],


    autopct="%1.1f%%"

)



plt.title(

    "Pollution Source Contribution Distribution"

)



plt.savefig(

    os.path.join(

        OUTPUT_FOLDER,

        "Source_Contribution_Pie.png"

    ),

    dpi=300

)



plt.close()



# ============================================================
# TEXT SUMMARY REPORT
# ============================================================


summary_text = f"""

AI URBAN AIR QUALITY INTELLIGENCE

GEOSPATIAL SOURCE ATTRIBUTION REPORT

Generated:
{datetime.now()}


Dominant Pollution Source:

{dominant_source["Pollution_Source"]}


Contribution:

{dominant_source["Contribution_%"]:.2f}%


Confidence:

{dominant_source["Confidence"]}



SOURCE CONTRIBUTION RANKING

"""


for _, row in final_report.iterrows():


    summary_text += (

        f"""

Rank {row['Rank']}

Source:
{row['Pollution_Source']}

Contribution:
{row['Contribution_%']:.2f}%

Confidence:
{row['Confidence']}


"""

    )





with open(

    os.path.join(

        OUTPUT_FOLDER,

        "Source_Attribution_Summary.txt"

    ),

    "w"

) as file:


    file.write(

        summary_text

    )



# ============================================================
# METADATA FILE
# ============================================================


metadata = f"""

PACKAGE:
10.1 Geospatial Pollution Source Attribution Engine


Model:
Production XGBoost AQI Model


Dataset Shape:
{X_test.shape}


Number of Features:
{len(X_test.columns)}


Number of Pollution Sources Detected:
{len(source_available_features)}


Generated:
{datetime.now()}

"""



with open(

    os.path.join(

        OUTPUT_FOLDER,

        "Package10_Metadata.txt"

    ),

    "w"

) as file:


    file.write(

        metadata

    )



# ============================================================
# FINAL OUTPUT
# ============================================================


print("\n============================================================")
print("FILES GENERATED")
print("============================================================")


print(

"""
1. Pollution_Source_Contribution.csv

2. Source_Confidence_Scores.csv

3. Source_Attribution_Report.csv

4. Feature_Level_Source_Contribution.csv

5. Frontend_Source_Attribution.json

6. Source_Contribution_Bar.png

7. Source_Contribution_Pie.png

8. Source_Attribution_Summary.txt

9. Package10_Metadata.txt
"""

)



print("\n============================================================")
print("PACKAGE 10.1 VERSION 2.0 COMPLETED SUCCESSFULLY")
print("============================================================")

PACKAGE 10.1 VERSION 2.0
GEOSPATIAL POLLUTION SOURCE ATTRIBUTION ENGINE

Configuration Loaded

Loading Test Dataset...

Dataset Shape
(182, 30)

Feature Count:
30

Loading Production XGBoost Model...
Model Loaded Successfully

Calculating SHAP Values...
SHAP Calculation Completed

SHAP Matrix Shape
(182, 30)

Detecting Available Features...

Detected Pollution Sources
Traffic Emissions: ['NO', 'NO2', 'NOx', 'CO', 'Benzene']
Particulate Matter: ['PM2.5', 'PM10', 'PM25_Diff', 'PM10_Diff', 'PM25_MA3', 'PM10_MA3', 'PM25_PM10_Ratio']
Meteorological Conditions: ['Rainfall', 'Humidity_Change', 'Temperature_Change', 'Wind_Speed', 'Wind_Change', 'Wind_Lag1', 'Wind_Humidity_Index']
Satellite Aerosols: ['MODIS_AOD', 'MODIS_AOD_MA7']
Industrial Pollution: ['SO2', 'NH3']
AQI Historical Pattern: ['AQI', 'AQI_Lag1', 'AQI_Lag7', 'AQI_MA3', 'AQI_MA7']

Calculating Source Contributions...
Source Contribution Calculation Completed

Normalizing Source Contributions...

POLLUTION SOURCE CONTRIBUTION
   Ran